# How LLMs Work Under the Hood
**40-minute presentation — interactive demos**

Requirements: `pip install ipywidgets`  
Works in JupyterLab, classic Notebook, and VS Code notebooks.

---

In [15]:
import math
import ipywidgets as widgets
from IPython.display import display

In [16]:
from pathlib import Path
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, logging as hf_logging

hf_logging.set_verbosity_error()

_MODEL_PATH = Path("~/models/huggingface/hub/models--openai-community--gpt2").expanduser()
_snapshot = next(d for d in (_MODEL_PATH / "snapshots").iterdir() if d.is_dir())

gpt2_tokenizer = GPT2Tokenizer.from_pretrained(_snapshot, local_files_only=True)
gpt2_model     = GPT2LMHeadModel.from_pretrained(_snapshot, local_files_only=True)
gpt2_model.eval()
print("GPT-2 loaded.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded.

---
## Part 1 — Text as Prediction

LLMs are next-token predictors. At each step the model produces a probability distribution over the vocabulary and samples from it. Step through the sentence below to see this token by token.

In [17]:
INITIAL_PROMPT = "The cat sat on the"
TOP_K = 5
BAR   = "█"
BAR_W = 30

def _get_candidates(token_ids):
    input_ids = torch.tensor([token_ids])
    with torch.no_grad():
        logits = gpt2_model(input_ids).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, TOP_K)
    return [
        (gpt2_tokenizer.decode([tid]).strip() or repr(gpt2_tokenizer.decode([tid])),
         tid,
         prob * 100)
        for prob, tid in zip(top_probs.tolist(), top_ids.tolist())
    ]

class NextTokenDemo:
    def __init__(self, initial_prompt=INITIAL_PROMPT):
        self._initial_ids = gpt2_tokenizer.encode(initial_prompt)
        self._token_ids   = list(self._initial_ids)
        self._candidates  = []

        self.prompt_input = widgets.Text(
            value=initial_prompt,
            placeholder="Enter a starting prompt…",
            continuous_update=False,
            layout=widgets.Layout(width="400px"),
        )
        self.set_btn    = widgets.Button(description="Set prompt", button_style="info")
        self.prompt_out = widgets.HTML()
        self.note_out   = widgets.HTML()

        self._btns = [widgets.Button(layout=widgets.Layout(width="110px")) for _ in range(TOP_K)]
        self._bars = [widgets.HTML() for _ in range(TOP_K)]
        for i, btn in enumerate(self._btns):
            btn.on_click(lambda _, idx=i: self._pick(idx))

        self.pick_btn  = widgets.Button(description="Pick most likely →", button_style="success")
        self.reset_btn = widgets.Button(description="Reset")
        self.set_btn.on_click(lambda _: self._set_prompt())
        self.prompt_input.observe(lambda _: self._set_prompt(), names="value")
        self.pick_btn.on_click(lambda _: self._pick(0))
        self.reset_btn.on_click(lambda _: self._reset())

        cand_rows = [widgets.HBox([b, r]) for b, r in zip(self._btns, self._bars)]
        self.ui = widgets.VBox([
            widgets.HTML("<b>Starting prompt:</b>"),
            widgets.HBox([self.prompt_input, self.set_btn]),
            widgets.HTML("<b>Sentence so far:</b>"),
            self.prompt_out,
            widgets.HTML("<b>Top candidates for next token:</b>"),
            widgets.VBox(cand_rows),
            widgets.HBox([self.pick_btn, self.reset_btn]),
            self.note_out,
        ])
        self._refresh()

    def _set_prompt(self):
        text = self.prompt_input.value.strip()
        if not text:
            return
        self._initial_ids = gpt2_tokenizer.encode(text)
        self._token_ids   = list(self._initial_ids)
        self._refresh()

    def _refresh(self):
        self.note_out.value = "<i>Thinking…</i>"
        self.pick_btn.disabled = True
        for btn in self._btns:
            btn.disabled = True
        self._candidates = _get_candidates(self._token_ids)
        self._render()

    def _render(self):
        prompt_text = gpt2_tokenizer.decode(self._token_ids)
        toks = prompt_text.split()
        hl = " ".join(
            f"<mark style='background:#dbeafe;padding:2px 6px;border-radius:4px'>{t}</mark>"
            if i == len(toks) - 1 else t
            for i, t in enumerate(toks)
        )
        self.prompt_out.value = (
            f"<span style='font-family:monospace;font-size:15px;line-height:2.2'>{hl}</span>"
        )

        max_pct = self._candidates[0][2]
        for i, (btn, bar) in enumerate(zip(self._btns, self._bars)):
            label, _, pct = self._candidates[i]
            bar_str = BAR * round(pct / max_pct * BAR_W)
            btn.description = label
            btn.style.button_color = "#f0fdf4" if i == 0 else "#f9fafb"
            btn.style.font_weight  = "bold" if i == 0 else "normal"
            btn.disabled = False
            bar.value = (
                f"<span style='font-family:monospace;line-height:2'>"
                f"{bar_str:<{BAR_W}} {pct:>5.1f}%</span>"
            )

        self.pick_btn.disabled = False
        self.note_out.value = "<i>Click any candidate to choose it, or use 'Pick most likely'.</i>"

    def _pick(self, idx):
        _, token_id, _ = self._candidates[idx]
        self._token_ids.append(token_id)
        self._refresh()

    def _reset(self):
        self._token_ids = list(self._initial_ids)
        self._refresh()

display(NextTokenDemo().ui)

---
## Part 2 — Attention: How Context Travels

Attention lets each token gather weighted information from every other token. The same word attends to very different neighbors depending on surrounding context — this is how "bank" (financial) vs "bank" (river) gets disambiguated.

**Q/K/V intuition:** Query = *what am I looking for?* · Key = *what do I contain?* · Value = *what do I contribute?*

In [18]:
ATTENTION_DATA = {
    "The river bank was steep after the flood": {
        "bank":  [("river",0.52),("flood",0.28),("steep",0.10),("The",0.05),("was",0.05)],
        "river": [("bank",0.44),("flood",0.30),("steep",0.15),("after",0.07),("The",0.04)],
        "flood": [("bank",0.48),("river",0.34),("after",0.10),("steep",0.05),("was",0.03)],
        "steep": [("bank",0.40),("river",0.32),("flood",0.16),("was",0.08),("The",0.04)],
    },
    "She could not bear to see the bear in the cage": {
        "bear (verb)": [("could",0.38),("not",0.29),("She",0.18),("to",0.10),("see",0.05)],
        "bear (noun)": [("cage",0.45),("see",0.32),("in",0.13),("the",0.06),("She",0.04)],
        "She":         [("bear(v)",0.41),("bear(n)",0.35),("could",0.14),("cage",0.07),("not",0.03)],
        "cage":        [("bear (noun)",0.55),("in",0.25),("see",0.12),("the",0.05),("She",0.03)],
    },
    "The model read the long document and summarized it": {
        "it":          [("document",0.58),("model",0.22),("summarized",0.12),("read",0.05),("long",0.03)],
        "summarized":  [("model",0.44),("read",0.30),("document",0.16),("and",0.07),("long",0.03)],
        "document":    [("long",0.47),("read",0.28),("summarized",0.15),("model",0.07),("the",0.03)],
        "model":       [("read",0.42),("summarized",0.35),("document",0.13),("long",0.07),("The",0.03)],
    },
}

NOTES = {
    "bank":        '"bank" strongly attends to "river" — context disambiguates meaning',
    "river":       '"river" cross-checks with "bank" and "flood"',
    "flood":       '"flood" links back to the nouns that caused it',
    "steep":       '"steep" attends to the subject nouns describing the terrain',
    "bear (verb)": 'First "bear" (endure) attends to modal "could not" — not an animal',
    "bear (noun)": 'Second "bear" attends strongly to "cage" — context resolves to an animal',
    "She":         '"She" tracks both uses of "bear" simultaneously',
    "cage":        '"cage" resolves the animal reading of "bear"',
    "it":          '"it" attends to "document" — pronoun resolution via attention',
    "summarized":  '"summarized" checks back to the agent ("model") and action ("read")',
    "document":    '"document" attends to its modifier "long" and verbs acting on it',
    "model":       '"model" anchors the subject role across the sentence',
}

BAR_W2 = 25

sent_sel = widgets.Dropdown(
    options=list(ATTENTION_DATA.keys()),
    description="Sentence:",
    layout=widgets.Layout(width="540px"),
    style={"description_width": "70px"},
)
tok_sel = widgets.ToggleButtons(
    options=list(ATTENTION_DATA[sent_sel.value].keys()),
    description="Token:",
    style={"description_width": "55px", "button_width": "130px"},
)
attn_out  = widgets.Output()
anote_out = widgets.HTML()

def render_attn(tok, sent):
    entries = ATTENTION_DATA[sent].get(tok, [])
    if not entries: return
    max_v = entries[0][1]
    lines = [f"{'Token':<16} {'Weight':>7}  Bar"]
    lines.append("-" * 50)
    for w, v in entries:
        bar = BAR * round(v / max_v * BAR_W2)
        lines.append(f"{w:<16} {v*100:>6.1f}%  {bar}")
    with attn_out:
        attn_out.clear_output(wait=True)
        print(f"Attention weights from '{tok}':\n")
        print("\n".join(lines))
    anote_out.value = f"<i style='font-size:13px'>{NOTES.get(tok,'')}</i>"

def on_sent(change):
    tok_sel.options = list(ATTENTION_DATA[change["new"]].keys())
    render_attn(tok_sel.value, change["new"])

sent_sel.observe(on_sent, names="value")
tok_sel.observe(lambda c: render_attn(c["new"], sent_sel.value), names="value")

render_attn(tok_sel.value, sent_sel.value)
display(widgets.VBox([
    sent_sel,
    widgets.HTML("<b>Select a token to see its attention weights:</b>"),
    tok_sel, attn_out, anote_out,
]))

---
## Part 3 — Probability Distributions During Inference

The model outputs a distribution over the vocabulary. **Temperature** controls how peaked or flat it is. **Top-p** (nucleus sampling) restricts sampling to the smallest set of tokens whose cumulative probability exceeds `p`.

Adjust both sliders to see the effect. Watch the entropy readout — it measures how "uncertain" the model is.

In [19]:
PROMPTS = {
    "The capital of France is ___": {
        "tokens": ["Paris","France","Lyon","Rome","Nice","London","Bordeaux"],
        "logits": [6.8, 3.1, 2.4, 1.9, 1.2, 0.9, 0.4],
    },
    "The best way to learn is ___": {
        "tokens": ["practice","doing","reading","by","repetition","failing","experimenting"],
        "logits": [4.2, 3.8, 3.4, 2.6, 2.1, 1.8, 1.2],
    },
    "Once upon a time there was ___": {
        "tokens": ["a","an","once","the","not","one","some"],
        "logits": [4.5, 3.9, 3.1, 2.8, 1.5, 1.2, 0.9],
    },
}

def softmax(logits):
    m = max(logits)
    e = [math.exp(x - m) for x in logits]
    s = sum(e)
    return [v/s for v in e]

def apply_temp(logits, t):
    return [l/t for l in logits]

def apply_top_p(probs, p):
    ranked = sorted(enumerate(probs), key=lambda x: -x[1])
    cum, keep = 0.0, set()
    for i, v in ranked:
        cum += v; keep.add(i)
        if cum >= p: break
    masked = [v if i in keep else 0.0 for i, v in enumerate(probs)]
    s = sum(masked)
    return [v/s for v in masked]

def entropy(probs):
    return -sum(p * math.log2(p) for p in probs if p > 0)

BAR_W3 = 24

prompt_sel = widgets.Dropdown(
    options=list(PROMPTS.keys()),
    description="Prompt:",
    layout=widgets.Layout(width="440px"),
    style={"description_width": "60px"},
)
temp_sl = widgets.FloatSlider(
    value=1.0, min=0.1, max=2.0, step=0.1,
    description="Temperature:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="440px"),
    readout_format=".1f",
)
topp_sl = widgets.FloatSlider(
    value=1.0, min=0.1, max=1.0, step=0.05,
    description="Top-p:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="440px"),
    readout_format=".2f",
)
dist_out  = widgets.Output()
dnote_out = widgets.HTML()

def render_dist(prompt_key, temp, p):
    data   = PROMPTS[prompt_key]
    tokens = data["tokens"]
    raw    = softmax(data["logits"])
    adj    = apply_top_p(softmax(apply_temp(data["logits"], temp)), p)
    max_r, max_a = max(raw), max(adj)
    top_i  = adj.index(max(adj))
    active = sum(1 for v in adj if v > 0.001)
    h      = entropy(adj)

    hdr = f"{'Token':<15} {'Raw':>7}  {'':^{BAR_W3}}  {'Adjusted':>9}  Adjusted bar"
    sep = "-" * (len(hdr) + 4)
    rows = [hdr, sep]
    for i, (tok, rp, ap) in enumerate(zip(tokens, raw, adj)):
        rb = BAR * round(rp / max_r * BAR_W3)
        ab = (BAR * round(ap / max_a * BAR_W3)) if ap > 0 else "(excluded)"
        tag = "  ◄ sampled" if i == top_i else ""
        rows.append(f"{tok:<15} {rp*100:>6.1f}%  {rb:<{BAR_W3}}  {ap*100:>8.1f}%  {ab}{tag}")
    rows += [sep, f"Entropy: {h:.2f} bits     Active tokens in nucleus: {active}"]

    with dist_out:
        dist_out.clear_output(wait=True)
        print("\n".join(rows))

    if temp < 0.5:
        note = "Low temperature: distribution peaks sharply — near-deterministic output."
    elif temp > 1.4:
        note = "High temperature: distribution flattens — more surprising completions."
    else:
        note = "Moderate temperature."
    if p < 1.0:
        note += f" Top-p={p:.2f} restricts sampling to the {active} tokens covering {p*100:.0f}% of probability mass."
    dnote_out.value = f"<i style='font-size:13px'>{note}</i>"

def on_dist_change(_):
    render_dist(prompt_sel.value, temp_sl.value, topp_sl.value)

for w in [prompt_sel, temp_sl, topp_sl]:
    w.observe(on_dist_change, names="value")

render_dist(prompt_sel.value, temp_sl.value, topp_sl.value)
display(widgets.VBox([prompt_sel, temp_sl, topp_sl, dist_out, dnote_out]))

---
## Part 4 — Putting It Together

The full inference loop:

> **tokens → embeddings → [attention + feedforward] × N layers → softmax → sample**

- Early layers handle syntax; later layers handle semantics and world knowledge.
- Training objective: predict next/masked tokens at massive scale. Weights that survive encode useful patterns.
- **What LLMs are not:** no persistent memory, no built-in retrieval, no logical reasoning engine.
- **What bolts on:** RAG, tool use, fine-tuning, RLHF — all steer the base prediction mechanism.

| Parameter | Low | High |
|---|---|---|
| Temperature | Consistent, deterministic | Creative, unpredictable |
| Top-p | Safest tokens only | Full vocabulary in play |
| Context length | Less to attend to | Richer attention signal |